<a href="https://colab.research.google.com/github/fatmasenguler/laplacian-minor-hierarchy/blob/main/compute_Rijkl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install biopython -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.8 MB/s eta 0:00:00


In [4]:
#!/usr/bin/env python3
# =============================================================================
#  R_ijkl Calculator — Fourth-Order Laplacian Minor Invariant
#  Laplacian Minor Hierarchy Framework (Senguler Ciftci & Erman 2026)
#
#  Computes R_ijkl for all l != i,j,k from a C-alpha contact network.
#
#  Mathematics
#  -----------
#  Let L be the weighted graph Laplacian and K = L^+ its Moore-Penrose
#  pseudoinverse. The effective distance/resistance is
#
#      R_ab = K_aa + K_bb - 2 K_ab.
#
#  This is equivalent to the spanning-tree minor formula
#
#      R_ab = det L(a,b) / tau,
#
#  because
#
#      det L(a,b) = tau * (K_aa + K_bb - 2 K_ab),
#
#  where tau is any cofactor determinant of L, i.e. the weighted spanning-tree
#  partition function. Therefore tau is NOT used to scale R_ab when K = L^+ is
#  already used. log(tau) is reported only as a diagnostic.
#
#  The fourth-order invariant is
#
#      R_ijkl = 1 - det G^(i)_{jkl} / (R_ij R_ik R_il),
#
#  where
#
#      G^(i)_{jkl} = [[R_ij,      K_jk^(i), K_jl^(i)],
#                    [K_jk^(i), R_ik,      K_kl^(i)],
#                    [K_jl^(i), K_kl^(i), R_il     ]]
#
#  and
#
#      K_ab^(i) = 0.5 * (R_ia + R_ib - R_ab).
#
#  Convention
#  ----------
#      R_ijkl = 0  uncorrelated
#      R_ijkl = 1  correlated
#
#  This matches the third-order index chi_ijk = K_jk^(i)^2 / (R_ij R_ik),
#  where chi = 0 is uncorrelated and chi = 1 is correlated.
#
#  Weighting schemes
#  -----------------
#      unweighted : w_ij = 1                  if d_ij <= cutoff
#      inv_d2     : w_ij = 1 / d_ij^2         if d_ij <= cutoff
#      exp        : w_ij = exp(-d_ij / kT)    if d_ij <= cutoff
#
#  For the exponential scheme, kT must have the same distance units as d_ij
#  because the exponent must be dimensionless. If kT is left blank in the
#  interactive prompt, the mean contact distance is used as a fallback.
# =============================================================================

from __future__ import annotations

import os
from typing import Optional

import numpy as np

try:
    from Bio.PDB import PDBParser
except ImportError as exc:
    raise ImportError("BioPython not found. Run:  !pip install biopython -q") from exc


# =============================================================================
#  GRAPH CONSTRUCTION
# =============================================================================

def build_contact_graph(
    pdb_path: str,
    cutoff: float = 7.8,
    chain_id: Optional[str] = None,
    scheme: str = "unweighted",
    kT: Optional[float] = None,
):
    """
    Parse a PDB file, extract C-alpha coordinates, and build a weighted Laplacian.

    Parameters
    ----------
    pdb_path : str
        Path to the PDB file.
    cutoff : float
        C-alpha distance cutoff in Angstrom.
    chain_id : str or None
        Chain ID to use. If None, all chains are used.
    scheme : str
        One of: "unweighted", "inv_d2", "exp".
    kT : float or None
        Exponential scale parameter for the exp scheme:

            w_ij = exp(-d_ij / kT),  if d_ij <= cutoff.

        kT must be positive and must have the same units as d_ij. If None,
        the mean contact distance is used as a fallback.

    Returns
    -------
    L : ndarray, shape (N, N)
        Weighted graph Laplacian.
    res_ids : list[int]
        PDB residue numbers in graph order.
    n : int
        Number of C-alpha residues.
    chain_used : str
        Chain label used for reporting.
    kT_used : float or None
        Exponential scale parameter used, if applicable.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_path)

    ca_coords = []
    res_ids = []
    chain_used = chain_id if chain_id else "all"

    # Use the first model only.
    model = next(structure.get_models())
    for chain in model:
        if chain_id is not None and chain.id != chain_id:
            continue
        for residue in chain:
            if "CA" in residue:
                ca_coords.append(residue["CA"].get_vector().get_array())
                res_ids.append(residue.get_id()[1])

    ca_coords = np.asarray(ca_coords, dtype=float)
    n = len(ca_coords)

    if n == 0:
        raise ValueError(
            f"No C-alpha atoms found. Check chain ID {chain_id!r} in {pdb_path!r}."
        )
    if n < 4:
        raise ValueError("At least 4 C-alpha residues are needed to compute R_ijkl.")

    # Pairwise C-alpha distances.
    diff = ca_coords[:, None, :] - ca_coords[None, :, :]
    D = np.sqrt(np.sum(diff * diff, axis=2))
    np.fill_diagonal(D, np.inf)

    contact = D <= cutoff
    if not np.any(contact):
        raise ValueError("No contacts found. Increase cutoff or check the PDB/chain.")

    if scheme == "unweighted":
        W = contact.astype(float)
        kT_used = None

    elif scheme == "inv_d2":
        W = np.where(contact, 1.0 / (D * D), 0.0)
        kT_used = None

    elif scheme == "exp":
        if kT is None:
            kT = float(np.mean(D[contact]))
        if kT <= 0:
            raise ValueError("kT must be positive for the exponential scheme.")
        W = np.where(contact, np.exp(-D / kT), 0.0)
        kT_used = float(kT)

    else:
        raise ValueError("Unknown scheme. Choose: unweighted | inv_d2 | exp")

    np.fill_diagonal(W, 0.0)

    # Weighted graph Laplacian: L_ii = sum_j w_ij, L_ij = -w_ij for i != j.
    L = -W
    np.fill_diagonal(L, np.sum(W, axis=1))

    return L, res_ids, n, chain_used, kT_used


# =============================================================================
#  LINEAR ALGEBRA
# =============================================================================

def is_connected_laplacian(L: np.ndarray, tol: float = 1e-14) -> bool:
    """Return True if the graph represented by Laplacian L is connected."""
    n = L.shape[0]
    adjacency = L < -tol

    seen = np.zeros(n, dtype=bool)
    stack = [0]
    seen[0] = True

    while stack:
        u = stack.pop()
        for v in np.flatnonzero(adjacency[u]):
            if not seen[v]:
                seen[v] = True
                stack.append(int(v))

    return bool(np.all(seen))


def compute_pseudoinverse_and_log_tau(L: np.ndarray):
    """
    Compute K = L^+ and log_tau = log(det L[1:,1:]).

    tau is diagnostic only. R_ab is computed directly from K = L^+ and is not
    divided by tau.
    """
    if L.ndim != 2 or L.shape[0] != L.shape[1]:
        raise ValueError("L must be a square matrix.")

    if not is_connected_laplacian(L):
        raise ValueError(
            "The contact graph is disconnected. Effective distances are not valid. "
            "Try increasing the cutoff or choosing a connected chain."
        )

    # L is symmetric/Hermitian. The hermitian flag improves numerical stability.
    try:
        K = np.linalg.pinv(L, hermitian=True)
    except TypeError:  # For older NumPy versions.
        K = np.linalg.pinv(L)

    # Remove tiny asymmetry from floating-point arithmetic.
    K = 0.5 * (K + K.T)

    # Compute log(tau) stably. Direct det() can overflow for large proteins.
    cofactor = L[1:, 1:]
    sign, log_tau = np.linalg.slogdet(cofactor)
    if sign <= 0:
        raise ValueError(
            "The Laplacian cofactor has non-positive determinant. "
            "Check graph connectivity and numerical conditioning."
        )

    return K, float(log_tau)


# =============================================================================
#  R_ijkl COMPUTATION
# =============================================================================

def effective_distance(K: np.ndarray, a: int, b: int) -> float:
    """Return R_ab = K_aa + K_bb - 2 K_ab."""
    val = float(K[a, a] + K[b, b] - 2.0 * K[a, b])

    # Clean up harmless roundoff. A true effective distance cannot be negative.
    if -1e-12 < val < 0.0:
        val = 0.0
    return val


def inner_product_at_ref(K: np.ndarray, a: int, b: int, ref: int) -> float:
    """Return K_ab^(ref) = 0.5 * (R_ref,a + R_ref,b - R_a,b)."""
    return 0.5 * (
        effective_distance(K, ref, a)
        + effective_distance(K, ref, b)
        - effective_distance(K, a, b)
    )


def compute_Rijkl(
    K: np.ndarray,
    i: int,
    j: int,
    k: int,
    l: int,
    denom_tol: float = 1e-14,
    value_tol: float = 1e-8,
) -> Optional[float]:
    """
    Compute the normalized fourth-order invariant R_ijkl.

    Returns
    -------
    float or None
        R_ijkl clipped only for tiny numerical roundoff outside [0,1]. Returns
        None if indices are not distinct, the denominator is too small, or a
        large numerical violation is detected.
    """
    if len({i, j, k, l}) < 4:
        return None

    Rij = effective_distance(K, i, j)
    Rik = effective_distance(K, i, k)
    Ril = effective_distance(K, i, l)

    denom = Rij * Rik * Ril
    if denom <= denom_tol:
        return None

    Kjk = inner_product_at_ref(K, j, k, i)
    Kjl = inner_product_at_ref(K, j, l, i)
    Kkl = inner_product_at_ref(K, k, l, i)

    G = np.array(
        [
            [Rij, Kjk, Kjl],
            [Kjk, Rik, Kkl],
            [Kjl, Kkl, Ril],
        ],
        dtype=float,
    )
    G = 0.5 * (G + G.T)

    val = float(1.0 - np.linalg.det(G) / denom)

    # The exact value should lie in [0, 1]. Only clip tiny roundoff.
    if val < -value_tol or val > 1.0 + value_tol:
        return None
    return float(np.clip(val, 0.0, 1.0))


# =============================================================================
#  INPUT/OUTPUT HELPERS
# =============================================================================

def residue_number_to_index(res_ids: list[int], pdb_number: int, label: str) -> int:
    """Map a PDB residue number to its 0-based index in res_ids."""
    matches = [idx for idx, rid in enumerate(res_ids) if rid == pdb_number]

    if not matches:
        raise ValueError(
            f"Residue {label}={pdb_number} not found. "
            f"Available PDB residue-number range: {min(res_ids)}-{max(res_ids)}"
        )

    if len(matches) > 1:
        raise ValueError(
            f"Residue number {pdb_number} occurs more than once. "
            "Specify a single chain ID instead of using all chains."
        )

    return matches[0]


def safe_tau_string(log_tau: float) -> str:
    """Format tau safely from log_tau."""
    if log_tau < 700.0:  # exp(709) is near the float64 overflow limit.
        return f"tau = {np.exp(log_tau):.6e}  (log tau = {log_tau:.4f})"
    return f"log tau = {log_tau:.4f}  (tau too large to print safely)"


def parse_optional_float(prompt: str) -> Optional[float]:
    """Read an optional positive float from stdin. Blank means None."""
    text = input(prompt).strip()
    if text == "":
        return None
    return float(text)


# =============================================================================
#  MAIN ROUTINE
# =============================================================================

def main():
    print("=" * 60)
    print("   R_ijkl Calculator — Laplacian Minor Hierarchy")
    print("=" * 60)

    pdb_path = input("\nPDB file path (e.g. 5HED.pdb): ").strip()
    if not os.path.isfile(pdb_path):
        raise FileNotFoundError(f"File not found: {pdb_path}")

    chain_id = input("Chain ID (e.g. A; blank = all chains): ").strip().upper() or None

    i_pdb = int(input("Residue i (PDB number): ").strip())
    j_pdb = int(input("Residue j (PDB number): ").strip())
    k_pdb = int(input("Residue k (PDB number): ").strip())

    if len({i_pdb, j_pdb, k_pdb}) < 3:
        raise ValueError("Residues i, j, and k must be distinct.")

    print("\nWeighting scheme:")
    print("  unweighted  — w_ij = 1")
    print("  inv_d2      — w_ij = 1/d^2")
    print("  exp         — w_ij = exp(-d/kT)")

    scheme = input("Choose scheme [unweighted]: ").strip().lower() or "unweighted"
    if scheme not in {"unweighted", "inv_d2", "exp"}:
        raise ValueError("Unknown scheme. Choose: unweighted | inv_d2 | exp")

    cutoff = 7.8
    kT = None
    if scheme == "exp":
        kT = parse_optional_float(
            "kT for exp(-d/kT) in Angstrom units "
            "[blank = mean contact distance]: "
        )
        if kT is not None and kT <= 0:
            raise ValueError("kT must be positive for the exponential scheme.")

    print(
        f"\nBuilding contact graph "
        f"(chain={chain_id or 'all'}, cutoff={cutoff} A, scheme={scheme}) ..."
    )
    L, res_ids, n, chain_used, kT_used = build_contact_graph(
        pdb_path=pdb_path,
        cutoff=cutoff,
        chain_id=chain_id,
        scheme=scheme,
        kT=kT,
    )

    n_contacts = int(np.sum(L < 0) // 2)
    print(f"  Chain     : {chain_used}")
    print(f"  Residues  : {n}  ({min(res_ids)}-{max(res_ids)})")
    print(f"  Contacts  : {n_contacts}")
    if scheme == "exp":
        print(f"  kT        : {kT_used:.6g} A")
        if kT is None:
            print("              blank input used mean contact distance")

    print("\nComputing K = L^+ and log(tau) ...")
    K, log_tau = compute_pseudoinverse_and_log_tau(L)
    print(f"  {safe_tau_string(log_tau)}")

    i_idx = residue_number_to_index(res_ids, i_pdb, "i")
    j_idx = residue_number_to_index(res_ids, j_pdb, "j")
    k_idx = residue_number_to_index(res_ids, k_pdb, "k")
    fixed = {i_idx, j_idx, k_idx}

    print(
        f"\nComputing R_ijkl "
        f"(i={i_pdb}, j={j_pdb}, k={k_pdb}, l=all valid residues) ...\n"
    )

    header = f"{'i':>6}  {'j':>6}  {'k':>6}  {'l':>6}  {'R_ijkl':>14}"
    divider = "-" * 50
    print(header)
    print(divider)

    results = []
    skipped = 0

    for l_idx, l_pdb in enumerate(res_ids):
        if l_idx in fixed:
            continue

        val = compute_Rijkl(K, i_idx, j_idx, k_idx, l_idx)
        if val is None:
            skipped += 1
            continue

        row = (i_pdb, j_pdb, k_pdb, l_pdb, val)
        results.append(row)
        print(f"{i_pdb:>6}  {j_pdb:>6}  {k_pdb:>6}  {l_pdb:>6}  {val:>14.6f}")

    print(divider)
    print(f"\nTotal computed : {len(results)}")
    if skipped:
        print(f"Skipped        : {skipped}")

    if results:
        all_vals = np.asarray([row[4] for row in results], dtype=float)
        min_idx = int(np.argmin(all_vals))
        max_idx = int(np.argmax(all_vals))
        print(f"\nMin  R_ijkl : {all_vals.min():.6f}  at l = {results[min_idx][3]}")
        print(f"Max  R_ijkl : {all_vals.max():.6f}  at l = {results[max_idx][3]}")
        print(f"Mean R_ijkl : {all_vals.mean():.6f}")

    protein_name = os.path.splitext(os.path.basename(pdb_path))[0]
    chain_tag = f"_chain{chain_used}" if chain_id else ""
    scheme_tag = scheme
    if scheme == "exp":
        scheme_tag = f"exp_kT{kT_used:.6g}".replace(".", "p")

    out_file = f"{protein_name}{chain_tag}_{scheme_tag}_Rijkl_i{i_pdb}_j{j_pdb}_k{k_pdb}.txt"

    with open(out_file, "w", encoding="utf-8") as f:
        f.write(
            f"# R_ijkl — {protein_name}  chain={chain_used}  "
            f"scheme={scheme}  cutoff={cutoff} A\n"
        )
        f.write(f"# i={i_pdb}  j={j_pdb}  k={k_pdb}\n")
        f.write(f"# log_tau = {log_tau:.12g}\n")
        if scheme == "exp":
            f.write(f"# weight = exp(-d/kT)\n")
            f.write(f"# kT = {kT_used:.12g} A\n")
            if kT is None:
                f.write("# kT_source = mean contact distance fallback\n")
        f.write(f"# {'i':>5}  {'j':>5}  {'k':>5}  {'l':>5}  {'R_ijkl':>14}\n")

        for row in results:
            f.write(
                f"  {row[0]:>5}  {row[1]:>5}  {row[2]:>5}  "
                f"{row[3]:>5}  {row[4]:>14.6f}\n"
            )

    print(f"\nResults saved to: {out_file}")
    print("=" * 60)

    return results


if __name__ == "__main__":
    main()

   R_ijkl Calculator — Laplacian Minor Hierarchy

PDB file path (e.g. 5HED.pdb): 5HED.pdb
Chain ID (e.g. A; blank = all chains): A
Residue i (PDB number): 330
Residue j (PDB number): 327
Residue k (PDB number): 372

Weighting scheme:
  unweighted  — w_ij = 1
  inv_d2      — w_ij = 1/d^2
  exp         — w_ij = exp(-d/kT)
Choose scheme [unweighted]: exp
kT for exp(-d/kT) in Angstrom units [blank = mean contact distance]: 1

Building contact graph (chain=A, cutoff=7.8 A, scheme=exp) ...
  Chain     : A
  Residues  : 118  (298-415)
  Contacts  : 535
  kT        : 1 A

Computing K = L^+ and log(tau) ...
  tau = 1.860595e-155  (log tau = -356.2798)

Computing R_ijkl (i=330, j=327, k=372, l=all valid residues) ...

     i       j       k       l          R_ijkl
--------------------------------------------------
   330     327     372     298        0.097033
   330     327     372     299        0.102502
   330     327     372     300        0.108376
   330     327     372     301        0.116